In [4]:
import pandas as pd
import numpy as np
import os
import glob 
import matplotlib.pyplot as plt

# Threshold vs Ion Count Rate
Detector Bias Kept to -2300 V

Threshold will vary from [-4mV, -8mV]

In [5]:
def read_mpa_file(filepath):
    realtime = None
    totalsum = None

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("REALTIME") and realtime is None:
                realtime = float(line.split()[1].strip())
            elif line.startswith("TOTALSUM") and totalsum is None:
                totalsum = float(line.split()[1].strip())

    return realtime, totalsum

def get_thres(filename, prefix):
    name_only = filename.replace(prefix, "").replace(".mpa", "")
    name_only = name_only.split("_")[0]
    return int(name_only)

def get_run(filename, prefix):
    name_only = filename.replace(prefix, "").replace(".mpa", "")
    if "_" in name_only:
        return int(name_only.split("_")[1])
    return 1

def build_df(data_folder, glob_pattern, prefix, exclude=None):
    rows = []
    for filepath in glob.glob(os.path.join(data_folder, glob_pattern)):
        filename = os.path.basename(filepath)
        if exclude and exclude in filename:
            continue  # skip files that belong to the other set

        threshold = get_thres(filename, prefix)
        run = get_run(filename, prefix)
        realtime, totalsum = read_mpa_file(filepath)
        rate = totalsum / realtime

        rows.append({
            "threshold": threshold,
            "run": run,
            "file": filename,
            "realtime_s": realtime,
            "totalsum": totalsum,
            "rate_hz": rate
        })
    return pd.DataFrame(rows)

data_folder = "ThresholdControl"

noise_df = build_df(data_folder, "TDCNoise*.mpa", prefix="TDCNoise")
tdc_df = build_df(data_folder, "TDC*.mpa", prefix="TDC", exclude="Noise")

In [6]:
noise_summary = (
    noise_df.groupby("threshold")
    .agg(total_counts=("totalsum", "sum"), total_time=("realtime_s", "sum"))
    .reset_index()
)

noise_summary["Noise"] = noise_summary["total_counts"] / noise_summary["total_time"]
noise_summary["Noise_err"] = np.sqrt(noise_summary["total_counts"]) / noise_summary["total_time"]

merged = pd.merge(tdc_df, noise_summary[["threshold", "Noise", "Noise_err"]], on="threshold")

merged["rate_err"] = np.sqrt(merged["totalsum"]) / merged["realtime_s"]
merged["rate_corrected"] = merged["rate_hz"] - merged["Noise"]
merged["rate_corrected_err"] = np.sqrt(merged["rate_err"]**2 + merged["Noise_err"]**2)

KeyError: 'threshold'